# Brand Review Consolidation

Merges the per-ASIN Amazon review exports for one product format (run here on Prenatal → Softgels) into a single workbook, tags each row with its brand, and de-duplicates it.

- **Input:** one `.xlsx` per ASIN named `<Brand>-<ASIN>.xlsx`; the brand is parsed from the filename.
- **Output:** `combined_matching_files.xlsx` and `combined_matching_files_no_duplicates.xlsx`, written back to the source folder.

To run against another product format, change `folder` in section 1.

In [1]:
from pathlib import Path
from openpyxl import load_workbook

In [2]:
import pandas as pd

## 1. Combining Review(s) files into one and adding brand names

First cell checks that every export shares the same column layout; the second merges only the files that pass and prepends `brand_name`.

In [33]:
folder = Path(r"C:\Users\fariz.ali\OneDrive - Utopia\hassan\Magnesium for Sleep\Magnesium for Sleep\To-Combine")
files = sorted(folder.glob("*.xlsx"))

if not files:
    raise FileNotFoundError(f"No .xlsx files found in {folder}")

def get_headers(file_path):
    # read_only streams the sheet instead of loading it; data_only returns cached values, not formulas
    workbook = load_workbook(filename=file_path, read_only=True, data_only=True)
    try:
        sheet = workbook[workbook.sheetnames[0]]  # first sheet only
        header_row = next(sheet.iter_rows(min_row=1, max_row=1, values_only=True), ())  # row 1 = header
        headers = [str(value).strip() for value in header_row if value is not None and str(value).strip()]
        return sheet.title, headers
    finally:
        workbook.close()

file_headers = []
for file_path in files:
    sheet_name, headers = get_headers(file_path)
    file_headers.append({
        "file_name": file_path.name,
        "sheet_name": sheet_name,
        "column_count": len(headers),
        "columns": headers,
    })

reference = file_headers[0]  # first file alphabetically defines the expected schema
reference_columns = reference["columns"]
reference_set = set(reference_columns)

results = []
for item in file_headers:
    current_columns = item["columns"]
    current_set = set(current_columns)
    results.append({
        "file_name": item["file_name"],
        "sheet_name": item["sheet_name"],
        "column_count": item["column_count"],
        "count_matches": item["column_count"] == len(reference_columns),
        "same_columns_same_order": current_columns == reference_columns,  # list compare: order-sensitive
        "same_columns_ignore_order": current_set == reference_set,        # set compare: order-blind
        "missing_columns": [col for col in reference_columns if col not in current_set],
        "extra_columns": [col for col in current_columns if col not in reference_set],
    })

print(f"Reference file: {reference['file_name']} | Sheet: {reference['sheet_name']}")
print(f"Reference column count: {len(reference_columns)}")
print("Reference columns:")
for idx, column in enumerate(reference_columns, start=1):
    print(f"{idx}. {column}")

# Check every row below: anything without count_matches=True and same_order=True is dropped by the next cell.
print("\nComparison summary:")
for row in results:
    print(
        f"- {row['file_name']} | count={row['column_count']} | "
        f"count_matches={row['count_matches']} | "
        f"same_order={row['same_columns_same_order']} | "
        f"same_set={row['same_columns_ignore_order']}"
    )
    if row['missing_columns']:
        print(f"  missing: {row['missing_columns']}")
    if row['extra_columns']:
        print(f"  extra: {row['extra_columns']}")


Reference file: Clean Nutraceuticals-B0BWBY6HFD.xlsx | Sheet: sheet1
Reference column count: 19
Reference columns:
1. Asin
2. Title
3. English Title
4. Content
5. English Content
6. Verified Purchase
7. Model
8. Rating
9. Helpful
10. Images
11. Images URL
12. Videos
13. Videos URL
14. Reviews URL
15. Author
16. Avatar URL
17. Author Homepage
18. Nation
19. Date

Comparison summary:
- Clean Nutraceuticals-B0BWBY6HFD.xlsx | count=19 | count_matches=True | same_order=True | same_set=True
- MEDCHOICE-B099JXNWYL.xlsx | count=19 | count_matches=True | same_order=True | same_set=True
- Natural Vitality-B085HK5LY2.xlsx | count=19 | count_matches=True | same_order=True | same_set=True
- Natural Vitality-B09345846G.xlsx | count=19 | count_matches=True | same_order=True | same_set=True
- Plus+Ultra-B0DY8698NC.xlsx | count=19 | count_matches=True | same_order=True | same_set=True
- Relaxium-B00NJ1BQZA.xlsx | count=19 | count_matches=True | same_order=True | same_set=True
- Smarter Nutrition-B07XJR

In [34]:
matching_files = [
    row['file_name']
    for row in results
    if row['count_matches'] and row['same_columns_same_order']
]

if not matching_files:
    raise ValueError('No files have matching column count and column order, so nothing can be combined.')

combined_rows = []
for file_name in matching_files:
    file_path = folder / file_name
    workbook = load_workbook(filename=file_path, read_only=True, data_only=True)
    try:
        sheet = workbook[workbook.sheetnames[0]]
        # 'Nature Made-B005DEK9FO.xlsx' -> 'Nature Made'; a hyphen inside a brand name would truncate it
        brand_name = file_path.stem.split('-', 1)[0].strip()
        for row in sheet.iter_rows(min_row=2, values_only=True):
            if all(value is None for value in row):
                continue
            # trim/pad to the reference width so a ragged row can't shift the columns
            values = list(row[:len(reference_columns)])
            if len(values) < len(reference_columns):
                values.extend([None] * (len(reference_columns) - len(values)))
            combined_rows.append([brand_name] + values)
    finally:
        workbook.close()

output_path = folder / 'combined_matching_files.xlsx'
# clone the first source file and wipe its rows, so the output inherits its column widths/formatting
output_workbook = load_workbook(filename=matching_files and (folder / matching_files[0]))
try:
    output_sheet = output_workbook.active
    output_sheet.title = 'combined'
    output_sheet.delete_rows(1, output_sheet.max_row)
    output_sheet.append(['brand_name'] + reference_columns)
    for row in combined_rows:
        output_sheet.append(row)
    output_workbook.save(output_path)
finally:
    output_workbook.close()

print(f"Combined file created: {output_path}")
print(f"Files combined: {len(matching_files)}")
print(f"Total records written: {len(combined_rows)}")


Combined file created: C:\Users\fariz.ali\OneDrive - Utopia\hassan\Magnesium for Sleep\Magnesium for Sleep\To-Combine\combined_matching_files.xlsx
Files combined: 9
Total records written: 12833


## 2. De-duplicating and saving

Reads the combined workbook back into pandas, drops duplicate rows, writes `combined_matching_files_no_duplicates.xlsx`, then re-reads it to confirm.

In [35]:
output_df = pd.read_excel(output_path)

In [36]:
output_path

WindowsPath('C:/Users/fariz.ali/OneDrive - Utopia/hassan/Magnesium for Sleep/Magnesium for Sleep/To-Combine/combined_matching_files.xlsx')

In [37]:
# no subset: compares all 20 columns, incl. the unique Reviews URL, so only verbatim re-scrapes match
# to de-duplicate on the review itself use e.g. subset=['Asin', 'Author', 'Content']
output_df.duplicated().sum()

np.int64(0)

In [38]:
output_df.drop_duplicates(inplace=True)

In [39]:
output_df.duplicated().sum()  # expect 0

np.int64(0)

In [40]:
output_file_path = folder  # unused leftover from an earlier draft; nothing below reads it

In [41]:
output_file_path_no_duplicates = folder / 'combined_matching_files_no_duplicates.xlsx'

In [42]:
output_file_path_no_duplicates

WindowsPath('C:/Users/fariz.ali/OneDrive - Utopia/hassan/Magnesium for Sleep/Magnesium for Sleep/To-Combine/combined_matching_files_no_duplicates.xlsx')

In [43]:
# index=False stops pandas writing an unnamed index column; overwrites the file if it already exists
output_df.to_excel(output_file_path_no_duplicates, index=False)

In [44]:
# round-trip check: re-read from disk to confirm the saved file is readable and clean
pd.read_excel(output_file_path_no_duplicates).duplicated().sum()

np.int64(0)

## 3. Unique ASINs

Listing unique ASINs inorder to append them in "C:\Users\fariz.ali\OneDrive - Utopia\Mubashir Imdad's files - PBI\Ingredient_Labels.xlsx" as With/Without Ingredient.

Sections 1–2 ran on Softgels; the cells below load the Capsules output, which is pulled in its own run.

In [45]:
# Capsules output (sections 1-2 ran on Softgels); hard-coded path, independent of the cells above
asin_df = pd.read_excel(output_file_path_no_duplicates)
asin_df

,brand_name,Asin,Title,English Title,Content,English Content,Verified Purchase,Model,Rating,Helpful,Images,Images URL,Videos,Videos URL,Reviews URL,Author,Avatar URL,Author Homepage,Nation,Date
0,Clean Nutraceuticals,B0BWBY6HFD,Works great,NaN,I like this product. It works well if you take...,NaN,Y,NaN,5,2.0,NaN,NaN,NaN,NaN,https://www.amazon.com/gp/customer-reviews/R3G...,Rabbit,NaN,NaN,US,2024-03-07 00:00:00
1,Clean Nutraceuticals,B0BWBY6HFD,Smaller Bottle,NaN,This arrived as just a bottle .. type 2 white ...,NaN,N,NaN,4,2.0,NaN,NaN,NaN,NaN,https://www.amazon.com/gp/customer-reviews/R3R...,4 Lil Lambs,NaN,NaN,US,2023-05-14 00:00:00
2,Clean Nutraceuticals,B0BWBY6HFD,Helps with racing mind at bedtime,NaN,"So far, it's working really well. Only used it...",NaN,Y,NaN,5,4.0,NaN,NaN,NaN,NaN,https://www.amazon.com/gp/customer-reviews/R2M...,crispycreme,NaN,NaN,US,2024-08-07 00:00:00
3,Clean Nutraceuticals,B0BWBY6HFD,Great product,NaN,"This is the third time I bought this item, it’...",NaN,Y,NaN,5,1.0,NaN,NaN,NaN,NaN,https://www.amazon.com/gp/customer-reviews/R3U...,Luis A. Zayas,NaN,NaN,US,2024-08-08 00:00:00
4,Clean Nutraceuticals,B0BWBY6HFD,Very good quality,NaN,Excellent sleeping supplement,NaN,Y,NaN,5,0.0,NaN,NaN,NaN,NaN,https://www.amazon.com/gp/customer-reviews/RS8...,Alejandra Cerdas,NaN,NaN,US,2025-05-15 08:00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12828,VEV,B0DFQ6SQ88,"Helps during menopause, AND to relax the body ...",NaN,Going through that of-so-fun time of life - me...,NaN,N,NaN,5,NaN,NaN,NaN,NaN,NaN,https://www.amazon.com/gp/customer-reviews/R2X...,ang,https://images-na.ssl-images-amazon.com/images...,https://www.amazon.com/gp/profile/amzn1.accoun...,US,2025-03-03 00:00:00
12829,VEV,B0DFQ6SQ88,Nice size; 2 are a dose,NaN,"VEV Magnesium Glycinate, Ashwagandha, L-Theani...",NaN,N,NaN,5,NaN,NaN,NaN,NaN,NaN,https://www.amazon.com/gp/customer-reviews/R33...,K & D,https://images-na.ssl-images-amazon.com/images...,https://www.amazon.com/gp/profile/amzn1.accoun...,US,2025-02-27 00:00:00
12830,VEV,B0DFQ6SQ88,Rated A,NaN,I liked this magnesium blend. It was easy on m...,NaN,Y,NaN,4,NaN,5.0,https://m.media-amazon.com/images/I/61HXdynzWI...,NaN,NaN,https://www.amazon.com/gp/customer-reviews/RJK...,JooLee,https://images-na.ssl-images-amazon.com/images...,https://www.amazon.com/gp/profile/amzn1.accoun...,US,2025-02-27 00:00:00
12831,VEV,B0DFQ6SQ88,Great quality! A really unique and helpful ble...,NaN,I have been taking these for a few weeks and r...,NaN,N,NaN,5,NaN,2.0,https://m.media-amazon.com/images/I/713H4jcAJU...,NaN,NaN,https://www.amazon.com/gp/customer-reviews/R25...,Hereby Grace,https://images-na.ssl-images-amazon.com/images...,https://www.amazon.com/gp/profile/amzn1.accoun...,US,2025-02-23 00:00:00


In [46]:
# dropna() skips blank cells; astype(str).str.strip() normalises type and stray whitespace first,
# so 'B0C6MZS7PG ' and 'B0C6MZS7PG' collapse to one value instead of counting as two
unique_asins = sorted(asin_df['Asin'].dropna().astype(str).str.strip().unique())

print(f"Unique ASINs: {len(unique_asins)}")
for asin in unique_asins:
    print(asin)

Unique ASINs: 15
B00NJ1BQZA
B0716M5CXM
B07XJRM3CZ
B07XKWRVN9
B07XPGMNBW
B085HK5LY2
B09345846G
B099JXNWYL
B09TZMSYX4
B0BKQ1V8PT
B0BL3VH4RZ
B0BWBY6HFD
B0CB83LD77
B0DFQ6SQ88
B0DY8698NC


In [47]:
# Copy straight to the clipboard instead of selecting the printed output by hand.
# index=False/header=False leaves one bare value per line, which Excel fills DOWN a column;
# a single-line clipboard is what makes Excel spread the values ACROSS a row instead.
pd.Series(unique_asins, name='Asin').to_clipboard(index=False, header=False)

print(f"{len(unique_asins)} ASINs on the clipboard - click the first Asin cell and press Ctrl+V")

15 ASINs on the clipboard - click the first Asin cell and press Ctrl+V
